# ARASO Factor Model - Research Notebook

**Adaptive Residual Arbitrage with Sentiment Overlay**

This notebook implements Phase 2: Factor Model + Signal Generation + Backtesting

## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# ARASO modules
import sys
sys.path.append('../strategies')
from araso_factor_model import (
    DatabaseConnector, 
    DynamicFactorModel, 
    FactorModelConfig,
    ARASOFactorPipeline
)
from araso_signals import (
    ARASOSignalGenerator, 
    ARASOConfig,
    MacroRegimeFilter
)
from araso_backtest import ARASOBacktester

%matplotlib inline
sns.set_style('whitegrid')

## 2. Database Connection

In [ ]:
# Initialize database connection
db = DatabaseConnector(
    username='your_username',  # Replace with actual
    password='your_password'   # Replace with actual
)
db.connect()
print("✅ Database connected")

## 3. Load Universe

In [ ]:
# Load active universe
pipeline = ARASOFactorPipeline(db)
universe = pipeline.load_universe()
print(f"Loaded {len(universe)} tickers")
print(f"First 10: {universe[:10]}")

## 4. Fetch Historical Data

In [ ]:
# Define backtest period
end_date = datetime(2024, 12, 31)
start_date = end_date - timedelta(days=252*5)  # 5 years

print(f"Fetching data from {start_date.date()} to {end_date.date()}")

# Fetch returns
returns_df = db.fetch_returns(universe, start_date, end_date)
print(f"Returns shape: {returns_df.shape}")
print(f"Date range: {returns_df.index[0]} to {returns_df.index[-1]}")

## 5. Factor Model - Single Date Test

In [ ]:
# Test factor model for a single date
test_date = datetime(2024, 12, 31)

results = pipeline.run_factor_model(test_date, lookback_days=126)

print(f"\nFactor Model Results for {test_date.date()}")
print(f"Variance explained: {results['explained_variance']:.2%}")
print(f"\nFactor returns (last 5 days):")
print(results['factor_returns'].tail())

## 6. Residual Analysis

In [ ]:
# Analyze residuals
residuals = results['residuals']
zscores = results['residual_zscores']

print("Residual Statistics:")
print(residuals.describe().iloc[:, :5])  # Show first 5 columns

# Plot residual distribution for AAPL
if 'AAPL' in residuals.columns:
    plt.figure(figsize=(12, 4))
    
    plt.subplot(1, 2, 1)
    residuals['AAPL'].hist(bins=30, alpha=0.7)
    plt.title('AAPL Residual Distribution')
    plt.xlabel('Residual Return')
    
    plt.subplot(1, 2, 2)
    zscores['AAPL'].hist(bins=30, alpha=0.7)
    plt.title('AAPL Z-Score Distribution')
    plt.xlabel('Z-Score')
    
    plt.tight_layout()
    plt.show()

## 7. HMM Regime Detection

In [ ]:
# Run HMM for universe
hmm_df = pipeline.run_hmm_for_universe(results['residuals'], test_date)

print(f"HMM results for {len(hmm_df)} stocks")
print("\nMean-reverting probability distribution:")
print(hmm_df['p_mean_reverting'].describe())

# Plot HMM state probabilities
plt.figure(figsize=(10, 4))
hmm_df['p_mean_reverting'].hist(bins=20, alpha=0.7)
plt.axvline(0.7, color='r', linestyle='--', label='Threshold (0.7)')
plt.title('Distribution of P(Mean-Reverting) Across Universe')
plt.xlabel('P(Mean-Reverting)')
plt.ylabel('Count')
plt.legend()
plt.show()

# Stocks with high mean-reversion probability
high_mr = hmm_df[hmm_df['p_mean_reverting'] > 0.8]
print(f"\n{len(high_mr)} stocks with P(MR) > 0.8:")
print(high_mr[['ticker', 'p_mean_reverting', 'residual_volatility']].head(10))

## 8. Signal Generation

In [ ]:
# Initialize signal generator
config = ARASOConfig(
    entry_threshold=1.5,
    exit_threshold=0.3,
    hmm_threshold=0.7
)
generator = ARASOSignalGenerator(config)

# Get macro regime
macro = generator.macro_filter.classify_regime(
    vix=18.5,
    yield_spread=0.8,
    cpi_surprise=0.1
)
print(f"Macro Regime: {macro.classification}")
print(f"Position Scale: {macro.position_scale}")

# Load universe info
universe_query = "SELECT ticker, sector FROM researcher.universe WHERE is_active=TRUE"
universe_df = pd.read_sql(universe_query, db.engine)

# Generate signals
signals = generator.generate_signals(
    test_date,
    results['residual_zscores'],
    hmm_df,
    macro,
    universe_df
)

print(f"\nGenerated {len(signals)} entry signals")
if len(signals) > 0:
    print("\nTop signals by confidence:")
    print(signals.nlargest(10, 'confidence_score')[
        ['ticker', 'direction', 'residual_zscore', 'hmm_mr_probability', 'confidence_score'
    ]])

## 9. Rolling Factor Model (Multi-Date)

In [ ]:
# Run factor model for multiple dates
# This would be the full backtest loop

rebalance_dates = pd.date_range(
    start='2024-01-01', 
    end='2024-12-31', 
    freq='W'  # Weekly rebalance
)

print(f"Running factor model for {len(rebalance_dates)} dates...")

all_factor_returns = []
all_residuals = []

for date in rebalance_dates[:5]:  # Limit to 5 for demo
    try:
        result = pipeline.run_factor_model(date, lookback_days=126)
        if result:
            all_factor_returns.append(result['factor_returns'])
            all_residuals.append(result['residuals'].iloc[-1])  # Latest only
    except Exception as e:
        print(f"Error on {date}: {e}")

print(f"Successfully processed {len(all_factor_returns)} dates")

## 10. Visualization

In [ ]:
# Plot factor returns over time
if all_factor_returns:
    factor_df = pd.concat(all_factor_returns)
    
    plt.figure(figsize=(14, 8))
    
    # Plot first 5 factors
    for i in range(min(5, factor_df.shape[1])):
        plt.subplot(2, 3, i+1)
        factor_df.iloc[:, i].plot()
        plt.title(f'Factor {i+1} Returns')
        plt.xlabel('Date')
        plt.ylabel('Return')
    
    plt.tight_layout()
    plt.show()
    
    # Cumulative factor returns
    plt.figure(figsize=(10, 6))
    (1 + factor_df).cumprod().plot()
    plt.title('Cumulative Factor Returns')
    plt.xlabel('Date')
    plt.ylabel('Cumulative Return')
    plt.legend(loc='upper left')
    plt.show()

## 11. Save Results to Database

In [ ]:
# Save single-date results
# pipeline.save_results(results, hmm_df)

print("To save results, uncomment the line above")
print("This will write to:")
print("  - researcher.factor_returns")
print("  - researcher.factor_loadings")
print("  - researcher.residual_returns")
print("  - researcher.hmm_states")

## 12. Next Steps

1. **Full Backtest**: Run factor model + signal generation for all dates
2. **Performance Analysis**: Calculate Sharpe, drawdown, win rate
3. **Sensitivity Analysis**: Test different parameter values
4. **Sentiment Integration**: Add RavenPack/Benzinga data
5. **Live Trading**: Connect to IBKR paper trading